In [1]:
import os
import openai
from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential
from azure.search.documents import SearchClient
from azure.search.documents.models import QueryType
from azure.identity import AzureDeveloperCliCredential
from azure.core.credentials import AzureKeyCredential

# Replace these with your own values, either in environment variables or directly here
load_dotenv()
AZURE_STORAGE_ACCOUNT = os.getenv("AZURE_STORAGE_ACCOUNT")
AZURE_STORAGE_CONTAINER = os.getenv("AZURE_STORAGE_CONTAINER")
AZURE_SEARCH_SERVICE = os.getenv("AZURE_SEARCH_SERVICE")
AZURE_SEARCH_KEY = os.getenv('AZURE_SEARCH_KEY')
AZURE_SEARCH_INDEX = os.getenv("AZURE_SEARCH_INDEX")
AZURE_TENANT_ID = os.getenv('TENANTID')
# AZURE_OPENAI_SERVICE = os.environ.get("AZURE_OPENAI_SERVICE")
AZURE_OPENAI_GPT_DEPLOYMENT = os.getenv("AZURE_OPENAI_GPT_DEPLOYMENT")
AZURE_OPENAI_CHATGPT_DEPLOYMENT = os.getenv("AZURE_OPENAI_CHATGPT_DEPLOYMENT")


OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

KB_FIELDS_CONTENT = os.getenv("KB_FIELDS_CONTENT") or "content"
KB_FIELDS_CATEGORY = os.getenv("KB_FIELDS_CATEGORY") or "category"
KB_FIELDS_SOURCEPAGE = os.getenv("KB_FIELDS_SOURCEPAGE") or "sourcepage"

# Use the current user identity to authenticate with Azure OpenAI, Cognitive Search and Blob Storage (no secrets needed, 
# just use 'az login' locally, and managed identity when deployed on Azure). If you need to use keys, use separate AzureKeyCredential instances with the 
# keys for each service

# azure_credential = DefaultAzureCredential()
azure_credential = AzureDeveloperCliCredential() if AZURE_TENANT_ID == None else AzureDeveloperCliCredential(tenant_id=AZURE_TENANT_ID, process_timeout=60)
search_creds = AzureKeyCredential(AZURE_SEARCH_KEY)

# Used by the OpenAI SDK
# openai.api_type = "azure"
# openai.api_base = f"https://{AZURE_OPENAI_SERVICE}.openai.azure.com"
# openai.api_version = "2022-12-01"

# Comment these two lines out if using keys, set your API key in the OPENAI_API_KEY environment variable instead
# openai.api_type = "azure_ad"
# openai.api_key = azure_credential.get_token("https://cognitiveservices.azure.com/.default").token

# Set up clients for Cognitive Search and Storage
search_client = SearchClient(
    endpoint=f"https://{AZURE_SEARCH_SERVICE}.search.windows.net",
    index_name=AZURE_SEARCH_INDEX,
    credential=search_creds)

In [2]:
# ChatGPT uses a particular set of tokens to indicate turns in conversations
# prompt_prefix = """<|im_start|>system
# You are an expert in Bank's policies that helps the company employees to answer policy questions. 
# Answer ONLY with the facts listed in the list of sources below. If there isn't enough information below, say you don't know. Do not generate answers that don't use the sources below. If asking a clarifying question to the user would help, ask the question. 
# Each source has a name followed by colon and the actual information, always include the source name for each fact you use in the response. Use square brakets to reference the source, e.g. [info1.txt]. Don't combine sources, list each source separately, e.g. [info1.txt][info2.pdf].

# Sources:
# {sources}

# <|im_end|>"""

# prompt_prefix = """<|im_start|>system
# You are an expert in policies, cooperation organization, salesforce that helps Inscale Technologies employees and customers to answer policy questions, how to use different applications and systems in the organization. 
# Answer the question as truthfully as possible using the provided sources, and if the answer is not contained within the text, then attempt to answer it. If asking a clarifying question to the user would help, ask the question. 
# Each source has a name followed by colon and the actual information, always include the source name for each fact you use in the response. Use square brakets to reference the source, e.g. [info1.txt]. Don't combine sources, list each source separately, e.g. [info1.txt][info2.pdf].

# Sources:
# {sources}

# <|im_end|>"""

prompt_prefix = """<|im_start|>system
You are an expert document title retriever. 
Answer the question as truthfully as possible using the provided sources, and if the answer is not contained within the text, then say Could'nt find Page Title.

Each source has a name followed by colon and the actual information, always include the source name for each fact you use in the response. Use square brakets to reference the source, e.g. [info1.txt]. Don't combine sources, list each source separately, e.g. [info1.txt][info2.pdf].

Sources:
{sources}

<|im_end|>"""

turn_prefix = """
<|im_start|>user
"""

turn_suffix = """
<|im_end|>
<|im_start|>assistant
"""

prompt_history = turn_prefix

history = []

summary_prompt_template = """Below is a summary of the conversation so far, and a new question asked by the user that needs to be answered by searching in a knowledge base. Generate a search query based on the conversation and the new question. Source names are not good search terms to include in the search query.

Summary:
{summary}

Question:
{question}

Search query:
"""

In [4]:
# Execute this cell multiple times updating user_input to accumulate chat history
user_input = "Please analyze the provided document and extract any underlined and uppercase texts. [00206B8EA8E0230426025452-0.pdf], [CamScanner 04-14-2023 13.52-0.pdf]."

# Exclude category, to simulate scenarios where there's a set of docs you can't see
exclude_category = None

if len(history) > 0:
    completion = openai.Completion.create(
        engine=AZURE_OPENAI_GPT_DEPLOYMENT,
        prompt=summary_prompt_template.format(summary="\n".join(history), question=user_input),
        temperature=0.7,
        max_tokens=32,
        stop=["\n"])
    search = completion.choices[0].text
else:
    search = user_input

# Alternatively simply use search_client.search(q, top=3) if not using semantic search
print("Searching:", search)
print("-------------------")
filter = "category ne '{}'".format(exclude_category.replace("'", "''")) if exclude_category else None
r = search_client.search(search, 
                         filter=filter,
                         query_type=QueryType.SEMANTIC, 
                         query_language="en-us", 
                         query_speller="lexicon", 
                         semantic_configuration_name="default", 
                         top=3)
results = [doc[KB_FIELDS_SOURCEPAGE] + ": " + doc[KB_FIELDS_CONTENT].replace("\n", "").replace("\r", "") for doc in r]
content = "\n".join(results)

prompt = prompt_prefix.format(sources=content) + prompt_history + user_input + turn_suffix

completion = openai.Completion.create(
    engine=AZURE_OPENAI_GPT_DEPLOYMENT, 
    prompt=prompt, 
    temperature=0.3, 
    max_tokens=2024,
    stop=["<|im_end|>", "<|im_start|>"])

prompt_history += user_input + turn_suffix + completion.choices[0].text + "\n<|im_end|>" + turn_prefix
history.append("user: " + user_input)
history.append("assistant: " + completion.choices[0].text)

print("\n-------------------\n".join(history))
print("\n-------------------\nPrompt:\n" + prompt)

Searching: Please analyze the provided document and extract any underlined and uppercase texts. [00206B8EA8E0230426025452-0.pdf], [CamScanner 04-14-2023 13.52-0.pdf].
-------------------
user: Please analyze the provided document and extract any underlined and uppercase texts. [00206B8EA8E0230426025452-0.pdf], [CamScanner 04-14-2023 13.52-0.pdf].
-------------------
assistant: 
From [00206B8EA8E0230426025452-0.pdf]:

- AN ORDER directing the 1ª - 19th Gamishee Banks to appear before this Honourable Court to show cause why they should not pay over to the Judgment Creditor the equivalent of the Judgment debt together with the cost the 10% interest from 14 July, 2017 - 27" June, 2022 when Judgment was delivered in favour of the Plaintiff/Judgment Creditor (A period of 5 years) in the sum of USD 2,500 (Two Thousand Five Hundred Dollars) being the interest accrued for the period of 5 years.

- AN ORDER NISI directing the 1* - 19th Garnishee Banks to appear before this Honourable Court to sh

In [ ]:
what are owner codes used for?
how can I setup AEX?
the step is not complete give more details
show sample using an image
How do I build a report in Salesforce
how do I create a quote in AvSight?


In [4]:
prompt

"<|im_start|>system\nYou are an expert in Bank's policies that helps the company employees to answer policy questions. \nAnswer ONLY with the facts listed in the list of sources below. If there isn't enough information below, say you don't know. Do not generate answers that don't use the sources below. If asking a clarifying question to the user would help, ask the question. \nEach source has a name followed by colon and the actual information, always include the source name for each fact you use in the response. Use square brakets to reference the source, e.g. [info1.txt]. Don't combine sources, list each source separately, e.g. [info1.txt][info2.pdf].\n\nSources:\nWhat are Owner Codes_-0.pdf: 4/25/23, 4:25 PMWhat are Owner Codes?ase)AvSight University (https://support.avsight.net/avsight-university/)Communication Hub (/support/s/project-management)My Account (/support/s/com -myWhat are Owner Codes?Jan 23, 2023 • KnowledgeTitleWhat are Owner Codes?Primary CategorySetupKB ContentWhat A

In [5]:
search

'what are owner codes used for?'

In [13]:
history

['user: what are owner codes used for?',
 'assistant: Owner codes are used to identify and control inventory within AvSight. They are also used to group and identify the inventory and its performance. They help to track the performance of a large purchase separately from other inventory, and can also be used to track consignment items. [What are Owner Codes_-0.pdf]',
 'user: how can I setup AEX?',
 'assistant: To setup AEX, you must first create an Authentication Detail Record in AvSight. This record authorizes the AvSight company to access the AEX service. To begin, click the Gear icon in the upper right side of the screen. On the left of the resulting screen, expand Custom Code and select Custom Metadata Types. From the list on the right, select Manage Records next to Authentication Detail. [AEX Setup-0.pdf]',
 'user: how can I setup AEX?',
 'assistant: To setup AEX, you must first create an Authentication Detail Record in AvSight. This record authorizes the AvSight company to access